# Editor compuesto y diseño

[00_indice](00_indice.ipynb) · [01_python_y_flet](01_python_y_flet.ipynb) · [02_modelos](02_modelos.ipynb) · [03_controles](03_controles.ipynb) · [04_graficas](04_graficas.ipynb) · [05_eventos](05_eventos.ipynb) · [06_referencia](06_referencia.ipynb) · [07_validacion_e_integracion](07_validacion_e_integracion.ipynb) · [08_conectar_algoritmo](08_conectar_algoritmo.ipynb) · [09_demo_engine_paso_a_paso](09_demo_engine_paso_a_paso.ipynb)

**AM / Evolución Lab · Flet 1.0.4 · Python 3**

PSO: [10_pso_simulacion_e_integracion](10_pso_simulacion_e_integracion.ipynb) · [11_pso_pruebas_y_analisis](11_pso_pruebas_y_analisis.ipynb)


## Dos vistas
Configuración agrupa parámetros y población. Resultados contiene métricas, gráfica, individuos actuales y recorrido. visible alterna vistas; volver a Configuración pausa reproducción. Fondo gris suave, tarjetas blancas y acento azul se definen en theme.py. ThemeMode.LIGHT activa tema claro; Theme(color_scheme_seed=ACCENT) armoniza controles Material.

## Menú principal, submenú y routers
Menu de views/menu.py permite cambiar entre algoritmos. MenuItem(route,label,icon) describe cada opción. Menu(items,on_navigate) recibe opciones y callback de selección; select(route) destaca el algoritmo activo con fondo azul. Esta clase no contiene Volver ni conoce las etapas de un algoritmo.

Submenu de views/submenu.py navega por Configuración, Simulación / Resultados y Comparar pruebas en PSO. SubmenuItem(route,label,icon) describe etapas. Submenu(items,on_navigate,on_back) recibe callbacks de navegación y regreso; set_enabled(route,enabled) controla acceso visual y select(route,can_go_back=False) destaca etapa y sincroniza flecha Volver. buttons almacena los botones por clave; back_button es la flecha. Su estilo es blanco con acento suave para distinguirlo del menú principal.

Las dos clases heredan Container por separado. Application.menu usa Menu y Dashboard.submenu usa Submenu. Cada nivel tiene un ViewRouter independiente: routes asocia controles a rutas; go(route) cambia visibilidad y conserva instancias; back() vuelve dentro de ese router. La flecha del Submenu sólo navega por etapas, sin cambiar de algoritmo. guards bloquea Resultados hasta tener datos; on_change sincroniza presentación. outlet aloja vistas y can_go_back consulta historial.

Al cambiar algoritmo se conservan parámetros, edición pendiente, resultados y etapa de cada panel, y se pausa reproducción. Para añadir un algoritmo se registra en OptimizationApp.panels y MenuItem. Para añadir una etapa se registra en Dashboard.routes y SubmenuItem.

## PopulationEditor
Es una clase derivada de ft.Container. Su contenido combina Column, ResponsiveRow, campos, botones y ListView. La etiqueta de cada coordenada aparece a la izquierda y el campo a la derecha.

| Parámetro | Valor predeterminado | Papel |
|---|---|---|
| dimensions | 2 | cantidad de coordenadas, entero positivo |
| labels | None | nombres; por defecto x1,x2,... |
| bounds | None | límites opcionales por coordenada; None permite cualquier real finito |
| on_change | None | callback que recibe copia de población tras cambios guardados |
| on_refresh | None | callback sin argumentos para solicitar repintado |
| list_height | 290 | altura de la lista desplazable |
| max_individuals | 200 | cantidad máxima |

El editor puede tener tres o más dimensiones. La app y las gráficas de ejemplo permiten sólo una o dos. labels y bounds deben tener dimensions elementos. Los límites son pares finitos y ascendentes.

## Métodos del editor
set_schema(dimensions,labels,bounds,notify=True) redefine campos y vacía población para evitar reinterpretar coordenadas. set_individuals(individuals,notify=True) reemplaza todos los valores después de validarlos; si falla, conserva la lista anterior. individuals devuelve una copia, no una referencia mutable interna.

save(e=None) agrega o modifica según editing_index. edit(index) carga valores exactos con repr, aunque la lista muestre cinco cifras significativas. delete(index) borra y reajusta una edición activa si se elimina una fila anterior. cancel_edit(e=None) descarta el formulario pendiente. on_submit de cada campo usa save; Enter equivale al botón.

_validate(values) convierte a float y comprueba coordenadas; _clear_form restablece campos; _render_list construye filas; _emit_population_change notifica; _refresh actualiza por callback o si está montado. Evitamos llamar _notify a nuestro método porque Flet usa ese nombre internamente.

## Controles y argumentos
Text(value,size,color,weight): contenido, tamaño, tinte y grosor. TextField(label,value,text_size,filled,fill_color,border,hint_text,expand,on_submit,on_change): entrada y estilo. border recibe OutlineInputBorder(border_radius=10,side=BorderSide(color=BORDER)); redondeo y color se definen en el objeto de borde. Se evita TextField.border_color y TextField.border_radius, obsoletos desde 1.0.0. Dropdown(label,value,options,on_select): selección única. DropdownOption(key,text): clave y etiqueta. ListView(height,spacing,build_controls_on_demand): lista acotada. Button(content,icon,on_click,elevation,bgcolor,color) define acción, relieve, fondo y texto. IconButton(icon,tooltip,icon_color,on_click): acciones por fila.

Container(content,padding,bgcolor,border,border_radius,col,width): estructura y estilo. Border.all(width,color): contorno. ResponsiveRow(spacing,run_spacing) divide en 12 columnas; col={'xs':12,'lg':4} cambia ancho por tamaño de pantalla. Row(wrap=True) permite partir la fila. Column(scroll=AUTO,expand=True) da desplazamiento a la página.

field(label,value='',**kwargs) aplica estilo compartido y pasa kwargs a TextField. card(title,subtitle,controls,col=12) construye tarjetas. No pases en kwargs una opción ya fijada por field sin adaptar ese helper.

[ListView](https://flet.dev/docs/controls/listview/) · [Container](https://flet.dev/docs/controls/container/)


In [ ]:
# Ejemplo de componente reutilizable; no monta una ventana.
from views.population_editor import PopulationEditor
editor = PopulationEditor(dimensions=3, labels=('Ancho','Alto','Peso'), bounds=((0,10),(0,10),(0,100)))
editor.set_individuals([(1,2,30)], notify=False)
print(editor.individuals)